In [1]:
import sagemaker
import boto3
from sagemaker.pytorch import PyTorch, PyTorchModel
import json

from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker.predictor import Predictor

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [ ]:
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "predictive-maintenance-data-1"

s3_train_path = f"s3://{bucket}/raw_dataset/conveyor_fault_dataset.csv"

print("Dataset location:", s3_train_path)

Dataset location: s3://relu-buckett/conveyor_fault_dataset.csv


In [3]:
estimator = PyTorch(
    entry_point="train.py",
    source_dir=".",
    role=role,
    framework_version="1.13",
    py_version="py39",
    instance_count=1,
    instance_type="ml.m5.xlarge",
    hyperparameters={
        "epochs": 20,
        "batch-size": 64,
        "hidden-size": 128,
        "num-layers": 2,
        "seq-len": 30,
        "lr": 1e-3
    },
)

estimator.fit({"train": s3_train_path})


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2025-10-19-20-59-55-122


2025-10-19 21:00:03 Starting - Starting the training job...
2025-10-19 21:00:18 Starting - Preparing the instances for training...
2025-10-19 21:01:03 Downloading - Downloading the training image.........
2025-10-19 21:02:14 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.9/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.9/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
2025-10-19 21:02:19,785 sagemaker-training-toolkit INFO     Imported f

In [4]:
role = sagemaker.get_execution_role()
session = sagemaker.Session()

model_artifact = estimator.model_data

pytorch_model = PyTorchModel(
    model_data=model_artifact,
    role=role,
    entry_point="inference.py",
    source_dir=".",
    framework_version="1.13",
    py_version="py39",
    env={
        'SAGEMAKER_MODEL_SERVER_TIMEOUT': '3600',
        'SAGEMAKER_MODEL_SERVER_WORKERS': '1',
        'SAGEMAKER_PROGRAM': 'inference.py'
    }
)

print("Deploying model...")
predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    wait=True
)

my_ml_endpoint =predictor.endpoint_name
print(f"Endpoint created: {predictor.endpoint_name}")


Deploying model...


INFO:sagemaker:Repacking model artifact (s3://sagemaker-eu-west-1-771826808190/pytorch-training-2025-10-19-20-59-55-122/output/model.tar.gz), script artifact (.), and dependencies ([]) into single tar.gz file located at s3://sagemaker-eu-west-1-771826808190/pytorch-inference-2025-10-19-21-03-16-032/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-inference-2025-10-19-21-03-19-911
INFO:sagemaker:Creating endpoint-config with name pytorch-inference-2025-10-19-21-03-20-506
INFO:sagemaker:Creating endpoint with name pytorch-inference-2025-10-19-21-03-20-506


------!Endpoint created: pytorch-inference-2025-10-19-21-03-20-506


In [ ]:
# Store the inference endpoint name in Parameter Store
import boto3

def store_endpoint_in_parameter_store(endpoint_name, parameter_name="/relu/sagemaker/inference-endpoint-name"):
    """
    Store the SageMaker endpoint name in AWS Systems Manager Parameter Store
    
    Args:
        endpoint_name (str): The name of the SageMaker endpoint
        parameter_name (str): The parameter store path
    """
    try:
        # Create SSM client
        ssm_client = boto3.client('ssm')
        
        # Store the endpoint name in parameter store
        response = ssm_client.put_parameter(
            Name=parameter_name,
            Value=endpoint_name,
            Type='String',
            Description='Name of the SageMaker inference endpoint for PMF model',
            Overwrite=True,
            Tags=[
                {
                    'Key': 'Environment',
                    'Value': 'prod'
                },
                {
                    'Key': 'Purpose',
                    'Value': 'SageMaker Configuration'
                }
            ]

        )
        
        print(f"✅ Successfully stored endpoint name in Parameter Store")
        print(f"   Parameter Name: {parameter_name}")
        print(f"   Endpoint Name: {endpoint_name}")
        print(f"   Version: {response.get('Version', 'N/A')}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error storing endpoint in Parameter Store: {str(e)}")
        return False

# Store the deployed endpoint name
print("Storing endpoint name in Parameter Store...")
success = store_endpoint_in_parameter_store(my_ml_endpoint)

if success:
    print("\n🎉 Endpoint name successfully stored in Parameter Store!")
    print("Other AWS services can now retrieve the endpoint name using:")
    print("  Parameter Path: /relu/sagemaker/inference-endpoint-name")
    print("  Value:", my_ml_endpoint)
else:
    print("\n⚠️  Failed to store endpoint name in Parameter Store")
    print("Please check your AWS credentials and permissions")

In [ ]:
# Helper function to retrieve endpoint name from Parameter Store
def get_endpoint_from_parameter_store(parameter_name="/relu/sagemaker/inference-endpoint-name"):
    """
    Retrieve the SageMaker endpoint name from AWS Systems Manager Parameter Store
    
    Args:
        parameter_name (str): The parameter store path
        
    Returns:
        str: The endpoint name if found, None otherwise
    """
    try:
        ssm_client = boto3.client('ssm')
        
        # Get the parameter value
        response = ssm_client.get_parameter(Name=parameter_name)
        endpoint_name = response['Parameter']['Value']
        
        print(f"✅ Successfully retrieved endpoint name from Parameter Store")
        print(f"   Parameter Name: {parameter_name}")
        print(f"   Endpoint Name: {endpoint_name}")
        print(f"   Last Modified: {response['Parameter']['LastModifiedDate']}")
        
        return endpoint_name
        
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ Parameter not found: {parameter_name}")
        return None
    except Exception as e:
        print(f"❌ Error retrieving parameter: {str(e)}")
        return None

# Example: Retrieve the endpoint name (for demonstration)
print("Testing parameter retrieval...")
retrieved_endpoint = get_endpoint_from_parameter_store()

if retrieved_endpoint:
    print(f"\n🔍 Retrieved endpoint name: {retrieved_endpoint}")
    print(f"🔄 Matches current endpoint: {retrieved_endpoint == my_ml_endpoint}")
else:
    print("\n⚠️  Could not retrieve endpoint name from Parameter Store")

In [9]:
session = sagemaker.Session()

endpoint_name = my_ml_endpoint
predictor = Predictor(endpoint_name=endpoint_name, sagemaker_session=session)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Configured predictor with JSON serializers")

print("\nTesting the endpoint...")

test_payload = {
    "instances": [
        {
            "Speed (rpm)": 120.0,
            "Load (kg)": 30.0,
            "Temperature (℃)": 40.0,
            "Vibration (m/s²)": 0.9,
            "Current (A)": 3.2
        },
        {
            "Speed (rpm)": 122.0,
            "Load (kg)": 525.0,
            "Temperature (℃)": 47.0,
            "Vibration (m/s²)": 0.95,
            "Current (A)": 3.43
        }
    ]
}

try:
    response = predictor.predict(test_payload)
    print("Success!")
    print("Response:", response)
    
    result = response
    print("\nPrediction Results:")
    for i, pred in enumerate(result["predictions"]):
        print(f"Sample {i+1}:")
        print(pred)
        # print(f"  Predicted Class: {pred['predicted_class']}")
        # print(f"  Confidence: {pred['confidence']:.4f}")
        # print(f"  All Probabilities: {pred['all_probabilities']}")
        print()
        
except Exception as e:
    print(f"Error: {e}")
    print("\nCheck CloudWatch logs for detailed error information:")
    print(f"https://console.aws.amazon.com/cloudwatch/home?region=us-west-2#logEventViewer:group=/aws/sagemaker/Endpoints/{endpoint_name}")

# Optional: Clean up (uncomment to delete endpoint after testing)
# print("\nCleaning up endpoint...")
# predictor.delete_endpoint()
# print("Endpoint deleted.")

Configured predictor with JSON serializers

Testing the endpoint...
Success!
Response: {'predictions': [{'predicted_class': 'belt slippage', 'predicted_class_id': 1, 'confidence': 0.9830097556114197, 'top_k': {'belt slippage': 0.9830097556114197, 'ball bearing': 0.013836823403835297, 'pulley': 0.0020434653852134943}, 'timestamp': '2025-10-19T21:55:10.721842Z'}, {'predicted_class': 'drive motor', 'predicted_class_id': 3, 'confidence': 0.6785109043121338, 'top_k': {'drive motor': 0.6785109043121338, 'central shaft': 0.17923401296138763, 'idler roller fault': 0.08219363540410995}, 'timestamp': '2025-10-19T21:55:10.721859Z'}]}

Prediction Results:
Sample 1:
{'predicted_class': 'belt slippage', 'predicted_class_id': 1, 'confidence': 0.9830097556114197, 'top_k': {'belt slippage': 0.9830097556114197, 'ball bearing': 0.013836823403835297, 'pulley': 0.0020434653852134943}, 'timestamp': '2025-10-19T21:55:10.721842Z'}

Sample 2:
{'predicted_class': 'drive motor', 'predicted_class_id': 3, 'confide